# FIGS — Feature Dependence

Model-agnostic **permutation importance** (AUC drop when a feature column is
shuffled) and **partial-dependence** curves for the top features. Works with the
mlx-boosting wrapper without needing native importances.

In [1]:
# Run with the `met` conda env kernel.
import warnings; warnings.filterwarnings('ignore')
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt
sys.path.insert(0, '..')   # so `import figs` works from notebooks/
from figs import config as C

In [2]:
from figs.data.dataset import read_dataset
from figs.model.wrapper import GBDTModel
from figs.config import LEAD_BANDS
from sklearn.metrics import roc_auc_score
from pathlib import Path
import json

DATA='../Data/processed/figs_v2.parquet'; MODELS='../Data/models'  # DATA must match the models
HAZARD='tor'; BAND=LEAD_BANDS[0].name; VAL_SAMPLE=40000
# feature list/order from the models' feature_cols.json (matches what they expect)
feats = json.loads((Path(MODELS)/'feature_cols.json').read_text())
# load this band's hazard model(s) ONCE — single file OR a bagged set (--bags)
bag_paths = (sorted(Path(MODELS).glob(f'hazard_{HAZARD}_{BAND}.pkl'))
             or sorted(Path(MODELS).glob(f'hazard_{HAZARD}_{BAND}_bag*.pkl')))
models = [GBDTModel.load(p) for p in bag_paths]
def predict_h(Xa): return np.mean([m.predict_pos(Xa) for m in models], axis=0)  # bag-mean

# stream ONLY this band's validation rows (sampled) — never the full matrix
b=[x for x in LEAD_BANDS if x.name==BAND][0]
va = read_dataset(DATA, columns=feats+[HAZARD], filters=[('fxx','>=',b.fmin),('fxx','<=',b.fmax),('split','==','validation')])
va = va.sample(min(len(va), VAL_SAMPLE), random_state=0)
X = va[feats].to_numpy('float32'); y = va[HAZARD].to_numpy(int)
base = roc_auc_score(y, predict_h(X)); print(f'{len(models)} bag(s) | baseline AUC', round(base,4))

3 bag(s) | baseline AUC 0.9812


## Permutation importance (top 25)

In [3]:
rng = np.random.default_rng(0)
imp = {}
for j,f in enumerate(feats):
    Xp = X.copy(); Xp[:,j] = rng.permutation(Xp[:,j])
    imp[f] = base - roc_auc_score(y, predict_h(Xp))
    if j % 100 == 0: print(f'{j}/{len(feats)}')
imp = pd.Series(imp).sort_values(ascending=False)
imp.head(25).iloc[::-1].plot.barh(figsize=(7,8)); plt.xlabel('AUC drop'); plt.title(f'{HAZARD} {BAND} permutation importance'); plt.show()
imp.head(25)

0/5086


KeyboardInterrupt: 

## Partial dependence for the top features

In [ ]:
top = imp.head(6).index.tolist()
fig, axes = plt.subplots(2,3, figsize=(15,8))
for ax,f in zip(axes.ravel(), top):
    j = feats.index(f)
    grid_vals = np.nanpercentile(X[:,j], np.linspace(2,98,20))
    pd_curve = []
    for g in grid_vals:
        Xt = X.copy(); Xt[:,j] = g; pd_curve.append(predict_h(Xt).mean())
    ax.plot(grid_vals, pd_curve, 'o-'); ax.set_title(f); ax.set_ylabel('mean p')
plt.tight_layout(); plt.show()